In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from utils.shared import write_to_table, notebook_exit
from utils.observability import create_control_table, insert_control_record
from utils.gold import (
    join_tables,
    get_station_status_facts,
    get_station_dim,
    get_weather_dim,
    add_station_facts_metadata,
)
from logger.custom_logging import set_up_logger, get_job_logger
from pyspark.sql.functions import max
import logging

try:
    context = REPLACED_WITH_SPARK_CONF
    run_id = context.tags().get("runId").get()
except:
    # Fallback for when running as a file (not a notebook)
    run_id = "unknown"

log_info = {
    "layer": "gold",
    "job": "join_station_and_weather_data",
    "dataset": "silver tables",
}

logger = set_up_logger()

log = get_job_logger(logger, **log_info, run_id=run_id)

log(logging.INFO, "Starting join_station_and_weather_data")

# The variables below are for the control table which is used to track the last processed timestamp and observe run metrics

start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
pipeline = "project_voltstream"
layer = f"{log_info["layer"]}.{log_info["job"]}"
error_message = None
records_processed = 0
records_failed = 0
last_processed_timestamp = None

try:
    control_table = create_control_table(spark)
    df_stations = spark.table("silver_dev.electrovolt.ev_stations")
    df_connections = spark.table("silver_dev.electrovolt.ev_connections")
    df_weather = spark.table("silver_dev.electrovolt.weather_data")
    df_joined = join_tables(df_stations, df_connections, df_weather)
    df_station_status_facts = get_station_status_facts(df_joined)
    df_station_status_facts = add_station_facts_metadata(df_station_status_facts)
    df_station_dim = get_station_dim(df_joined)
    df_weather_dim = get_weather_dim(df_joined)
    write_to_table(
        df_station_status_facts,
        "gold_dev.electrovolt.station_facts",
        "append",
        **log_info,
    )
    write_to_table(
        df_station_dim, "gold_dev.electrovolt.station_dim", "overwrite", **log_info
    )
    write_to_table(
        df_weather_dim, "gold_dev.electrovolt.weather_dim", "overwrite", **log_info
    )
    status = "success"
    records_processed = df_joined.count()
    last_processed_timestamp = df_station_status_facts.agg(
        max("ingest_timestamp")
    ).collect()[0][0]
    log(logging.INFO, "Finished process_station_connector_data")
    log(logging.INFO, "Finished join_station_and_weather_data")

except Exception as e:
    status = "failed"
    error_message = str(e)
    raise

end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
insert_control_record(
    spark,
    control_table,
    pipeline,
    layer,
    last_processed_timestamp,
    records_processed,
    records_failed,
    start_time,
    end_time,
    error_message,
    status,
)
notebook_exit("SUCCESS")